# Recurrent Neural Networks for South Park Character Classification\n
\n
This notebook implements and compares three recurrent neural network architectures:\n
\n
1. **Simple RNN** - Basic recurrent neural network\n
2. **LSTM** - Long Short-Term Memory network\n
3. **GRU** - Gated Recurrent Unit

## 1. Imports and Setup

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

import pandas as pd
import numpy as np
import re
from collections import Counter
from typing import List, Dict, Tuple

from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import accuracy_score, f1_score

import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import time

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Configuration Parameters

In [ ]:
K = 12
EMBEDDING_DIM = 64
HIDDEN_DIM = 128
NUM_LAYERS = 2
DROPOUT_RATE = 0.5
BIDIRECTIONAL = True
BATCH_SIZE = 64
LEARNING_RATE = 0.001
NUM_EPOCHS = 50
WEIGHT_DECAY = 1e-4
EARLY_STOP_PATIENCE = 7
MIN_WORD_FREQ = 2
MAX_VOCAB_SIZE = 10000
VAL_SIZE = 0.15
TEST_SIZE = 0.15

## 3. Load Data (Same as Baseline)

In [ ]:
# Load South Park dialogue data
base_url = "https://raw.githubusercontent.com/BobAdamsEE/SouthParkData/refs/heads/master/by-season/Season-{}.csv"

print("Loading South Park dialogue data...")
dfs = []
for season in tqdm(range(1, 20), desc="Loading seasons"):
    url = base_url.format(season)
    dfs.append(pd.read_csv(url))

df = pd.concat(dfs, ignore_index=True)
print(f"Loaded {len(df):,} dialogue lines")

## 4. Data Cleaning and Character Selection

In [ ]:
# Clean data
df_clean = df.drop_duplicates()
df_clean = df_clean.dropna(subset=['Character', 'Line'])
df_clean['Character'] = df_clean['Character'].str.strip()
df_clean['Line'] = df_clean['Line'].str.strip()
df_clean = df_clean[df_clean['Line'].str.len() > 0]

# Select top K characters
character_counts = df_clean['Character'].value_counts()
top_k_characters = character_counts.head(K).index.tolist()

print(f"Top {K} characters:")
for i, char in enumerate(top_k_characters, 1):
    count = character_counts[char]
    print(f"  {i:2d}. {char:20s} - {count:5d} lines")

# Filter dataset
df_model = df_clean[df_clean['Character'].isin(top_k_characters)].copy()
print(f"Filtered dataset: {len(df_model):,} lines")

## 5. Create Label Mappings and Split Data

In [ ]:
# Create mappings
char_to_label = {char: idx for idx, char in enumerate(top_k_characters)}
label_to_char = {idx: char for char, idx in char_to_label.items()}
df_model['label'] = df_model['Character'].map(char_to_label)

texts = df_model['Line'].tolist()
labels = df_model['label'].tolist()

# Split data
train_val_texts, test_texts, train_val_labels, test_labels = train_test_split(
    texts, labels, test_size=TEST_SIZE, stratify=labels, random_state=42
)

val_size_adjusted = VAL_SIZE / (1 - TEST_SIZE)
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_val_texts, train_val_labels, test_size=val_size_adjusted, 
    stratify=train_val_labels, random_state=42
)

print(f"Train: {len(train_texts):,}  Val: {len(val_texts):,}  Test: {len(test_texts):,}")

## 6. Text Preprocessing and Vocabulary

In [ ]:
def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\\s]', '', text)
    return text.split()

# Preprocess
processed_train_texts = [preprocess_text(t) for t in tqdm(train_texts, desc="Train")]
processed_val_texts = [preprocess_text(t) for t in tqdm(val_texts, desc="Val")]
processed_test_texts = [preprocess_text(t) for t in tqdm(test_texts, desc="Test")]

# Build vocabulary
class Vocabulary:
    def __init__(self, min_freq=1, max_size=None):
        self.word2idx = {'<pad>': 0, '<unk>': 1}
        self.idx2word = {0: '<pad>', 1: '<unk>'}
        self.min_freq = min_freq
        self.max_size = max_size
    
    def build_vocab(self, texts):
        word_counts = Counter(word for text in texts for word in text)
        word_counts = {w: c for w, c in word_counts.items() if c >= self.min_freq}
        sorted_words = sorted(word_counts.items(), key=lambda x: x[1], reverse=True)
        if self.max_size:
            sorted_words = sorted_words[:self.max_size - 2]
        for word, _ in sorted_words:
            idx = len(self.word2idx)
            self.word2idx[word] = idx
            self.idx2word[idx] = word
    
    def encode(self, text):
        return [self.word2idx.get(word, self.word2idx['<unk>']) for word in text]
    
    def __len__(self):
        return len(self.word2idx)

vocab = Vocabulary(min_freq=MIN_WORD_FREQ, max_size=MAX_VOCAB_SIZE)
vocab.build_vocab(processed_train_texts)
print(f"Vocabulary size: {len(vocab):,}")

# Encode
indexed_train_texts = [vocab.encode(t) for t in processed_train_texts]
indexed_val_texts = [vocab.encode(t) for t in processed_val_texts]
indexed_test_texts = [vocab.encode(t) for t in processed_test_texts]

## 7. Dataset and DataLoader

In [ ]:
class DialogueDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return {
            'text': torch.tensor(self.texts[idx], dtype=torch.long),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
            'length': len(self.texts[idx])
        }

def collate_fn(batch):
    batch = sorted(batch, key=lambda x: x['length'], reverse=True)
    max_len = batch[0]['length']
    texts, labels, lengths = [], [], []
    for item in batch:
        text = item['text']
        padded = torch.cat([text, torch.zeros(max_len - len(text), dtype=torch.long)])
        texts.append(padded)
        labels.append(item['label'])
        lengths.append(item['length'])
    return {
        'text': torch.stack(texts),
        'label': torch.stack(labels),
        'length': torch.tensor(lengths, dtype=torch.long)
    }

# Create datasets and loaders
train_dataset = DialogueDataset(indexed_train_texts, train_labels)
val_dataset = DialogueDataset(indexed_val_texts, val_labels)
test_dataset = DialogueDataset(indexed_test_texts, test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

print(f"Train batches: {len(train_loader)}  Val batches: {len(val_loader)}  Test batches: {len(test_loader)}")

## 8. Import RNN Models

In [ ]:
from rnn_models import RNNClassifier, LSTMClassifier, GRUClassifier

# Initialize models
rnn_model = RNNClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, K, NUM_LAYERS, DROPOUT_RATE, BIDIRECTIONAL)
lstm_model = LSTMClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, K, NUM_LAYERS, DROPOUT_RATE, BIDIRECTIONAL)
gru_model = GRUClassifier(len(vocab), EMBEDDING_DIM, HIDDEN_DIM, K, NUM_LAYERS, DROPOUT_RATE, BIDIRECTIONAL)

print(f"RNN parameters:  {sum(p.numel() for p in rnn_model.parameters()):,}")
print(f"LSTM parameters: {sum(p.numel() for p in lstm_model.parameters()):,}")
print(f"GRU parameters:  {sum(p.numel() for p in gru_model.parameters()):,}")

## 9. Class Weights

In [ ]:
class_weights = compute_class_weight('balanced', classes=np.arange(K), y=train_labels)
class_weights = torch.tensor(class_weights, dtype=torch.float).to(device)
print("Class weights computed")

## 10. Training Function with Early Stopping

In [ ]:
def train_model(model, model_name, train_loader, val_loader, num_epochs, lr, wd, class_weights, patience=7):
    model.to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_loss = float('inf')
    best_model_state = None
    patience_counter = 0
    
    print(f"Training {model_name}...")
    start_time = time.time()
    
    for epoch in range(num_epochs):
        # Train
        model.train()
        train_loss = 0.0
        train_preds, train_labels_list = [], []
        
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
            text = batch['text'].to(device)
            labels = batch['label'].to(device)
            lengths = batch['length']
            
            optimizer.zero_grad()
            outputs = model(text, lengths)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            preds = torch.argmax(outputs, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels_list.extend(labels.cpu().numpy())
        
        train_loss /= len(train_loader)
        train_acc = accuracy_score(train_labels_list, train_preds)
        
        # Validate
        model.eval()
        val_loss = 0.0
        val_preds, val_labels_list = [], []
        
        with torch.no_grad():
            for batch in val_loader:
                text = batch['text'].to(device)
                labels = batch['label'].to(device)
                lengths = batch['length']
                outputs = model(text, lengths)
                loss = criterion(outputs, labels)
                val_loss += loss.item()
                preds = torch.argmax(outputs, dim=1)
                val_preds.extend(preds.cpu().numpy())
                val_labels_list.extend(labels.cpu().numpy())
        
        val_loss /= len(val_loader)
        val_acc = accuracy_score(val_labels_list, val_preds)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        
        print(f"Epoch {epoch+1}: Train Loss={train_loss:.4f} Val Loss={val_loss:.4f} Val Acc={val_acc:.4f}")
        
        # Early stopping
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            patience_counter = 0
        else:
            patience_counter += 1
        
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
    
    model.load_state_dict(best_model_state)
    elapsed = time.time() - start_time
    print(f"Training completed in {elapsed/60:.2f} minutes")
    
    return {'model': model, 'history': history, 'best_val_loss': best_val_loss, 'time': elapsed}

## 11. Train All Models

Now train RNN, LSTM, and GRU. This may take 10-30 minutes depending on your hardware.

In [ ]:
# Train RNN
rnn_results = train_model(rnn_model, "RNN", train_loader, val_loader, NUM_EPOCHS, LEARNING_RATE, WEIGHT_DECAY, class_weights, EARLY_STOP_PATIENCE)

In [ ]:
# Train LSTM
lstm_results = train_model(lstm_model, "LSTM", train_loader, val_loader, NUM_EPOCHS, LEARNING_RATE, WEIGHT_DECAY, class_weights, EARLY_STOP_PATIENCE)

In [ ]:
# Train GRU
gru_results = train_model(gru_model, "GRU", train_loader, val_loader, NUM_EPOCHS, LEARNING_RATE, WEIGHT_DECAY, class_weights, EARLY_STOP_PATIENCE)

## 12. Evaluate and Compare Models

Evaluate all three models on the test set and compare their performance.

In [ ]:
def evaluate_rnn(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            text = batch['text'].to(device)
            labels = batch['label'].to(device)
            lengths = batch['length']
            outputs = model(text, lengths)
            preds = torch.argmax(outputs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    return {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, average='macro', zero_division=0),
        'recall': recall_score(all_labels, all_preds, average='macro', zero_division=0),
        'f1_score': f1_score(all_labels, all_preds, average='macro', zero_division=0)
    }

# Evaluate
rnn_test = evaluate_rnn(rnn_model, test_loader)
lstm_test = evaluate_rnn(lstm_model, test_loader)
gru_test = evaluate_rnn(gru_model, test_loader)

# Print comparison
print("=" * 80)
print("TEST SET PERFORMANCE COMPARISON")
print("=" * 80)
print(f"{'Model':<10} {'Accuracy':<12} {'Precision':<12} {'Recall':<12} {'F1-Score':<12}")
print("-" * 80)
print(f"{'RNN':<10} {rnn_test['accuracy']:<12.4f} {rnn_test['precision']:<12.4f} {rnn_test['recall']:<12.4f} {rnn_test['f1_score']:<12.4f}")
print(f"{'LSTM':<10} {lstm_test['accuracy']:<12.4f} {lstm_test['precision']:<12.4f} {lstm_test['recall']:<12.4f} {lstm_test['f1_score']:<12.4f}")
print(f"{'GRU':<10} {gru_test['accuracy']:<12.4f} {gru_test['precision']:<12.4f} {gru_test['recall']:<12.4f} {gru_test['f1_score']:<12.4f}")
print("=" * 80)

## 13. Save Best Model

In [ ]:
# Determine best model
best_name, best_f1, best_model = max(
    [('RNN', rnn_test['f1_score'], rnn_model),
     ('LSTM', lstm_test['f1_score'], lstm_model),
     ('GRU', gru_test['f1_score'], gru_model)],
    key=lambda x: x[1]
)

print(f"Best model: {best_name} (F1={best_f1:.4f})")

# Save checkpoint
checkpoint = {
    'model_type': best_name,
    'model_state_dict': best_model.state_dict(),
    'config': {
        'K': K, 'vocab_size': len(vocab), 'embedding_dim': EMBEDDING_DIM,
        'hidden_dim': HIDDEN_DIM, 'num_layers': NUM_LAYERS,
        'dropout_rate': DROPOUT_RATE, 'bidirectional': BIDIRECTIONAL
    },
    'char_to_label': char_to_label,
    'label_to_char': label_to_char,
    'test_metrics': {'rnn': rnn_test, 'lstm': lstm_test, 'gru': gru_test}
}

torch.save(checkpoint, f'best_rnn_model_{best_name.lower()}_k{K}.pt')
print(f"Checkpoint saved to: best_rnn_model_{best_name.lower()}_k{K}.pt")